# 0. Libraries

In [9]:
import json
import pandas as pd
from utilities.extract_qids import extract_qids

from utilities.alignment import (
    align_sections,
    align_sections_sentence_best_median,
    display_alignment,
    save_alignment,
)

from utilities.load_models import load_Qwen3_8B_via_llama_cpp
from utilities.omission_3v2 import calculate_omission,load_qwen_embedding
from utilities.propositioner_3v2 import extract_propositions_V3
from utilities.translate_json import translate_json, replace_ap_es_text


# 1. Wikipedia content download

In [10]:

data_en = extract_qids(
    "en",
    "Falkland_Islands",
    verbose=False,
) 
# data_en and data_es are lists of dictionaries
data_es = extract_qids(
    "es",
    "Islas_Malvinas_(territorio_británico_de_ultramar)",
    verbose=False,
)

with open( "data_es.json", "w", encoding="utf-8" ) as f:
    json.dump(data_es, f, indent=4, ensure_ascii=False)
    print("Triples saved to data_es.json")

with open( "data_en.json", "w", encoding="utf-8" ) as f:
    json.dump(data_en, f, indent=4, ensure_ascii=False)
    print("Triples saved to data_en.json")




   batch 1/11 (25 titles)
   batch 2/11 (25 titles)
   batch 3/11 (25 titles)
   batch 4/11 (25 titles)
   batch 5/11 (25 titles)
   batch 6/11 (25 titles)
   batch 7/11 (25 titles)
   batch 8/11 (25 titles)
   batch 9/11 (25 titles)
   batch 10/11 (25 titles)
   batch 11/11 (14 titles)
   batch 1/11 (25 titles)
   batch 2/11 (25 titles)
   batch 3/11 (25 titles)
   batch 4/11 (25 titles)
   batch 5/11 (25 titles)
   batch 6/11 (25 titles)
   batch 7/11 (25 titles)
   batch 8/11 (25 titles)
   batch 9/11 (25 titles)
   batch 10/11 (25 titles)
   batch 11/11 (14 titles)
   batch 1/7 (25 titles)
   batch 2/7 (25 titles)
   batch 3/7 (25 titles)
   batch 4/7 (25 titles)
   batch 5/7 (25 titles)
   batch 6/7 (25 titles)
   batch 7/7 (8 titles)
   batch 1/7 (25 titles)
   batch 2/7 (25 titles)
   batch 3/7 (25 titles)
   batch 4/7 (25 titles)
   batch 5/7 (25 titles)
   batch 6/7 (25 titles)
   batch 7/7 (8 titles)
Triples saved to data_es.json
Triples saved to data_en.json


# 2. Section alignment

In [11]:
# ALIGNMENT METHOD
# Choose "manual" or "sentence_best_median".
ALIGNMENT_METHOD = "sentence_best_median"

MIN_SCORE = 0.80

if ALIGNMENT_METHOD == "manual":
    alignment_results = align_sections(
        article_a=data_es,
        article_b=data_en,
        min_score=MIN_SCORE,
    )
    output_path = (
        "results/"
        "falkland_malvinas_alignment_manual.json"
    )

elif ALIGNMENT_METHOD == "sentence_best_median":
    alignment_results = align_sections_sentence_best_median(
        article_a=data_es,
        article_b=data_en,
        min_score=MIN_SCORE,
    )
    output_path = (
        "results/"
        "falkland_malvinas_alignment_best_sentence_median.json"
    )

else:
    raise ValueError(
        f"Unknown alignment method: {ALIGNMENT_METHOD!r}"
    )

display_alignment(alignment_results)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 14887.09it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 12840.77it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEX


SECTION ALIGNMENTS
es:Q9648:0 <-> en:Q9648:0  es:Q9648:0+en:Q9648:0+Best Sentence Median  Best Sentence Median  score=0.8626
es:Q9648:1 <-> en:Q9648:1  es:Q9648:1+en:Q9648:1+Best Sentence Median  Best Sentence Median  score=0.8829
es:Q9648:2 <-> en:Q9648:2  es:Q9648:2+en:Q9648:2+Best Sentence Median  Best Sentence Median  score=0.8746
es:Q9648:3 <-> en:Q9648:3  es:Q9648:3+en:Q9648:3+Best Sentence Median  Best Sentence Median  score=0.8727
es:Q9648:4 <-> en:Q9648:5  es:Q9648:4+en:Q9648:5+Best Sentence Median  Best Sentence Median  score=0.8555
es:Q9648:5 <-> en:Q9648:6  es:Q9648:5+en:Q9648:6+Best Sentence Median  Best Sentence Median  score=0.8596
es:Q9648:6 <-> en:Q9648:7  es:Q9648:6+en:Q9648:7+Best Sentence Median  Best Sentence Median  score=0.9225
es:Q9648:7 <-> en:Q9648:9  es:Q9648:7+en:Q9648:9+Best Sentence Median  Best Sentence Median  score=0.8581


In [12]:
alignment_results

[{'section_id_a': 'es:Q9648:0',
  'section_id_b': 'en:Q9648:0',
  'pair_id': 'es:Q9648:0+en:Q9648:0+Best Sentence Median',
  'alignment_score': 'Best Sentence Median',
  'numeric_alignment_score': 0.8626},
 {'section_id_a': 'es:Q9648:1',
  'section_id_b': 'en:Q9648:1',
  'pair_id': 'es:Q9648:1+en:Q9648:1+Best Sentence Median',
  'alignment_score': 'Best Sentence Median',
  'numeric_alignment_score': 0.8829},
 {'section_id_a': 'es:Q9648:2',
  'section_id_b': 'en:Q9648:2',
  'pair_id': 'es:Q9648:2+en:Q9648:2+Best Sentence Median',
  'alignment_score': 'Best Sentence Median',
  'numeric_alignment_score': 0.8746},
 {'section_id_a': 'es:Q9648:3',
  'section_id_b': 'en:Q9648:3',
  'pair_id': 'es:Q9648:3+en:Q9648:3+Best Sentence Median',
  'alignment_score': 'Best Sentence Median',
  'numeric_alignment_score': 0.8727},
 {'section_id_a': 'es:Q9648:4',
  'section_id_b': 'en:Q9648:5',
  'pair_id': 'es:Q9648:4+en:Q9648:5+Best Sentence Median',
  'alignment_score': 'Best Sentence Median',
  'numer

# 3. Proposition extraction

In [ ]:
qwen_model = load_Qwen3_8B_via_llama_cpp()
# Extract the two articles

In [8]:
propositions_list_en_qwen = []

for section in data_en["sections"]:
    propositions=extract_propositions_V3(
                section=section,
                model=qwen_model,
                language=section["language"]
            )

        # extend instead of append so that the final JSON
        # contains one flat list of propositions.

    propositions_list_en_qwen.extend(propositions)

    # -----------------------------------------------------------------
    # Save JSON
    # -----------------------------------------------------------------

with open("propositions_en_sentence_level_qwen.json", "w", encoding="utf-8") as file:
    json.dump( propositions_list_en_qwen, file, indent=4, ensure_ascii=False)

print("-----------------------------------")
print("Atomization completed")
print(f"Total propositions: {len(propositions_list_en_qwen)}")
print(f"Saved to: propositions_en_sentence_level_qwen.json")
print("-----------------------------------")



Processing section: en:Q9648:0
Number of sentences: 20
Processing sentence 1/20
Propositions for sentence: 3
Processing sentence 2/20
Propositions for sentence: 3
Processing sentence 3/20
Propositions for sentence: 5
Processing sentence 4/20
Propositions for sentence: 4
Processing sentence 5/20
Propositions for sentence: 3
Processing sentence 6/20
Propositions for sentence: 2
Processing sentence 7/20
Propositions for sentence: 2
Processing sentence 8/20
Propositions for sentence: 4
Processing sentence 9/20
Propositions for sentence: 2
Processing sentence 10/20
Propositions for sentence: 1
Processing sentence 11/20
Propositions for sentence: 2
Processing sentence 12/20
Propositions for sentence: 1
Processing sentence 13/20
Propositions for sentence: 2
Processing sentence 14/20
Propositions for sentence: 4
Processing sentence 15/20
Propositions for sentence: 2
Processing sentence 16/20
Propositions for sentence: 1
Processing sentence 17/20
Propositions for sentence: 2
Processing sentenc

In [9]:
propositions_list_es_qwen = []

for section in data_es["sections"]:
    propositions=extract_propositions_V3(
                section=section,
                model=qwen_model,
                language=section["language"]
            )

        # extend instead of append so that the final JSON
        # contains one flat list of propositions.

    propositions_list_es_qwen.extend(propositions)

    # -----------------------------------------------------------------
    # Save JSON
    # -----------------------------------------------------------------

with open("propositions_es_sentence_level_qwen.json", "w", encoding="utf-8") as file:
    json.dump( propositions_list_es_qwen, file, indent=4, ensure_ascii=False)

print("-----------------------------------")
print("Atomization completed")
print(f"Total propositions: {len(propositions_list_es_qwen)}")
print(f"Saved to: propositions_es_sentence_level_qwen.json")
print("-----------------------------------")


Processing section: es:Q9648:0
Number of sentences: 7
Processing sentence 1/7
Propositions for sentence: 5
Processing sentence 2/7
Propositions for sentence: 3
Processing sentence 3/7
Propositions for sentence: 2
Processing sentence 4/7
Propositions for sentence: 2
Processing sentence 5/7
Propositions for sentence: 2
Processing sentence 6/7
Propositions for sentence: 4
Processing sentence 7/7
Propositions for sentence: 3
Total propositions: 21

Processing section: es:Q9648:1
Number of sentences: 5
Processing sentence 1/5
Propositions for sentence: 2
Processing sentence 2/5
Propositions for sentence: 2
Processing sentence 3/5
Propositions for sentence: 3
Processing sentence 4/5
Propositions for sentence: 4
Processing sentence 5/5
Propositions for sentence: 0
Total propositions: 11

Processing section: es:Q9648:2
Number of sentences: 38
Processing sentence 1/38
Propositions for sentence: 1
Processing sentence 2/38
Propositions for sentence: 2
Processing sentence 3/38
Propositions for se

In [13]:
with open("propositions_en_sentence_level_qwen.json", "r", encoding="utf-8") as file:
    propositions_list_en_qwen = json.load(file)

with open("propositions_es_sentence_level_qwen.json", "r", encoding="utf-8") as file:
    propositions_list_es_qwen = json.load(file)
    propositions_list_es_qwen=translate_json(propositions_list_es_qwen)

Translating: 100%|██████████| 19/19 [00:17<00:00,  1.11it/s]


# 4. Omission Analysis

In [10]:
# manual
list_manual_section_pairs = [
    {
        "section_id_a": "es:Q9648:0",
        "section_id_b": "en:Q9648:0",
        "pair_id": "es:Q9648:0+en:Q9648:0+Manual",
        "alignment_score": "Manual"
    },
    {
        "section_id_a": "es:Q9648:1",
        "section_id_b": "en:Q9648:1",
        "pair_id": "es:Q9648:1+en:Q9648:1+Manual",
        "alignment_score": "Manual"
    },
    {
        "section_id_a": "es:Q9648:2",
        "section_id_b": "en:Q9648:2",
        "pair_id": "es:Q9648:2+en:Q9648:2+Manual",
        "alignment_score": "Manual"
    },
    {
        "section_id_a": "es:Q9648:3",
        "section_id_b": "en:Q9648:3",
        "pair_id": "es:Q9648:3+en:Q9648:3+Manual",
        "alignment_score": "Manual"
    },
    {
        "section_id_a": "es:Q9648:5",
        "section_id_b": "en:Q9648:6",
        "pair_id": "es:Q9648:5+en:Q9648:6+Manual",
        "alignment_score": "Manual"
    },
    {
        "section_id_a": "es:Q9648:6",
        "section_id_b": "en:Q9648:7",
        "pair_id": "es:Q9648:6+en:Q9648:7+Manual",
        "alignment_score": "Manual"
    }
]

In [5]:
tokenizer,model_embedding,device=load_qwen_embedding()

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 5618.61it/s]


In [11]:


results_omission=[]

for pair in list_manual_section_pairs:
    section_pair_id=pair["pair_id"]
    section_id_en=pair["section_id_b"]
    section_id_es=pair["section_id_a"]

    section_aps_en=[
        {
            "proposition_id":ap["proposition_id"],
            "text":ap["text"]
        }
        for ap in propositions_list_en_qwen
        if section_id_en == ap["section_id"]
    ]

    section_aps_es=[
        {
            "proposition_id":ap["proposition_id"],
            "text":ap["text_en"]
        }
        for ap in propositions_list_es_qwen
        if section_id_es == ap["section_id"]
    ]

    print(
        f"{section_id_en} (EN) vs {section_id_es} (ES) | "
        f"EN: {len(section_aps_en)} | ES: {len(section_aps_es)}"
    )

    omission_metrics=calculate_omission(
        section_aps_en,
        section_aps_es,
        threshold=0.70,
        tokenizer=tokenizer,
        model=model_embedding,
        device=device
    )

    results_omission.append({
        "section_pair_id":section_pair_id,
        "section_en":{
            "section_id":section_id_en,
            "propositions":section_aps_en
        },
        "section_es":{
            "section_id":section_id_es,
            "propositions":section_aps_es
        },
        "omission_metrics":omission_metrics
    })

with open("omission_results_qwen.json","w",encoding="utf-8") as file:
    json.dump(
        results_omission,
        file,
        ensure_ascii=False,
        indent=4
    )


en:Q9648:0 (EN) vs es:Q9648:0 (ES) | EN: 53 | ES: 21
en:Q9648:1 (EN) vs es:Q9648:1 (ES) | EN: 28 | ES: 11
en:Q9648:2 (EN) vs es:Q9648:2 (ES) | EN: 147 | ES: 92
en:Q9648:3 (EN) vs es:Q9648:3 (ES) | EN: 61 | ES: 45
en:Q9648:6 (EN) vs es:Q9648:5 (ES) | EN: 67 | ES: 19
en:Q9648:7 (EN) vs es:Q9648:6 (ES) | EN: 47 | ES: 39


# 5. Non-omitted AP equivalents

In [12]:
equivalent_aps = {}


for section in results_omission:
    section_pair_id = section["section_pair_id"]
    metrics = section["omission_metrics"]

    # ---------------------------------------------------------
    # EN -> ES
    # ---------------------------------------------------------

    for match_group in metrics["en_to_es"]["matches"]:
        ap_en = match_group["ap"]

        for match in match_group["matches"]:
            ap_es = match["ap"]

            key = (
                section_pair_id,
                ap_en["proposition_id"],
                ap_es["proposition_id"]
            )

            equivalent_aps[key] = {
                "section_pair_id": section_pair_id,
                "ap_en_id": ap_en["proposition_id"],
                "ap_en_text": ap_en["text"],
                "ap_es_id": ap_es["proposition_id"],
                "ap_es_text": ap_es["text"],
                "similarity": match["similarity"]
            }


    # ---------------------------------------------------------
    # ES -> EN
    # ---------------------------------------------------------

    for match_group in metrics["es_to_en"]["matches"]:
        ap_es = match_group["ap"]

        for match in match_group["matches"]:
            ap_en = match["ap"]

            key = (
                section_pair_id,
                ap_en["proposition_id"],
                ap_es["proposition_id"]
            )

            # Solo añadir si no estaba ya en EN -> ES
            if key not in equivalent_aps:
                equivalent_aps[key] = {
                    "section_pair_id": section_pair_id,
                    "ap_en_id": ap_en["proposition_id"],
                    "ap_en_text": ap_en["text"],
                    "ap_es_id": ap_es["proposition_id"],
                    "ap_es_text": ap_es["text"],
                    "similarity": match["similarity"]
                }


# Convertir a lista
equivalent_aps = list(equivalent_aps.values())
equivalent_aps = replace_ap_es_text(equivalent_aps,propositions_list_es_qwen)

with open("equivalent_aps_qwen.json", "w", encoding="utf-8") as f:
    json.dump(equivalent_aps, f, ensure_ascii=False, indent=4)